# 03 LoRA Finetuning

Qwen2-0.5B LoRA ??????????

???? GitHub ??????????????????????????


## Source: lora_model.ipynb


In [ ]:
import sys
import torch
import os
from transformers import (
    Trainer,
    TrainingArguments,
    DataCollatorForSeq2Seq,
    AutoModelForCausalLM,
    AutoTokenizer
)
# 其他必要的库（如数据集加载、数据处理等）


In [ ]:
import os
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"


In [ ]:
# Cell 1: 基础导入
import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForSeq2Seq,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, TaskType, get_peft_model
from swanlab.integration.huggingface import SwanLabCallback
  # 用于在 SwanLab 看板上监控训练进度


In [ ]:
# Cell 2 改成这样：加载并切分数据集（重命名列）
import pandas as pd
from datasets import Dataset

# 切换到你的项目目录

# 1. 读入原始 JSON
df = pd.read_json('poems_clean.json')


# 2. 只保留 translation → input, content → output
df = df[['translation', 'content']].rename(
    columns={'translation': 'input', 'content': 'output'}
)

# 3. 转成 HuggingFace Dataset 并 9:1 切分
ds = Dataset.from_pandas(df)
ds = ds.train_test_split(test_size=0.1, seed=42)
train_ds = ds['train']
eval_ds  = ds['test']

print(f"Train size: {len(train_ds)}, Eval size: {len(eval_ds)}")


In [ ]:
# Cell 3: 加载 tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    'Qwen/Qwen2-0.5B',
    use_fast=False,
    trust_remote_code=True
)

In [ ]:
# Cell 4: 数据预处理
def process_func(example):
    MAX_LENGTH = 600
    # system+user+assistant 三段式 prompt
    prompt = (
        "<|im_start|>system\n"
        "你是一名古诗词诗人，擅长将白话文转换为古诗词风格。\n"
        "<|im_end|>\n"
        "<|im_start|>user\n"
        f"{example['input']}\n"
        "<|im_end|>\n"
        "<|im_start|>assistant\n"
    )
    resp = example['output']
    p_tok = tokenizer(prompt, add_special_tokens=False)
    r_tok = tokenizer(resp,   add_special_tokens=False)
    input_ids      = p_tok.input_ids + r_tok.input_ids + [tokenizer.pad_token_id]
    attention_mask = p_tok.attention_mask + r_tok.attention_mask + [1]
    labels = [-100] * len(p_tok.input_ids) + r_tok.input_ids + [tokenizer.pad_token_id]
    # 截断
    if len(input_ids) > MAX_LENGTH:
        input_ids      = input_ids[:MAX_LENGTH]
        attention_mask = attention_mask[:MAX_LENGTH]
        labels         = labels[:MAX_LENGTH]
    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

# 应用到 train/eval
tokenized = ds.map(process_func, remove_columns=ds['train'].column_names, batched=False)
train_dataset = tokenized['train']
eval_dataset  = tokenized['test']

In [ ]:
# Cell 5: 加载本地模型并添加 LoRA
from transformers import AutoModelForCausalLM
from peft import LoraConfig, TaskType, get_peft_model

model = AutoModelForCausalLM.from_pretrained(
    './Qwen2-0.5B',             # ← 本地路径
    device_map="auto",         
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    local_files_only=True       # ← 强制使用本地模型文件
)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    inference_mode=False,
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


In [ ]:
# Cell 6: 设置训练参数
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./output/qwen2-0.5b-poetry-lora",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    logging_steps=5,
    eval_strategy="steps",  # 修改这里：evaluation_strategy -> eval_strategy
    eval_steps=50,
    num_train_epochs=5,
    save_steps=100,
    learning_rate=1e-4,
    gradient_checkpointing=True,
    fp16=True,
    report_to="none"
)


In [ ]:
# Cell 7: Trainer 并启动训练
from transformers import Trainer  # 确保导入 Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True),
    # callbacks=[SwanCallback()],  # 已移除未定义的 SwanCallback
)

# 清理 GPU 缓存（可选，根据显存情况决定是否需要）
torch.cuda.empty_cache()

# 启动训练
trainer.train()

In [ ]:
# Cell 8: 推理示例
from peft import PeftModel

# 重新加载基础模型 + LoRA 权重
base_model = AutoModelForCausalLM.from_pretrained(
    'Qwen/Qwen2-0.5B',
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True
).eval()
model_inf = PeftModel.from_pretrained(base_model, './output/qwen2-0.5b-poetry-lora/checkpoint-100')

# 构造 prompt
plain_text = "春风又绿江南岸，明月何时照我还？"
chat_input = (
    "<|im_start|>system\n你是一名古诗词诗人，将白话文转成古诗词。\n<|im_end|>\n"
    "<|im_start|>user\n" + plain_text + "\n<|im_end|>\n"
    "<|im_start|>assistant\n"
)
inputs = tokenizer(chat_input, return_tensors="pt").to('cuda')
outputs = model_inf.generate(**inputs, max_length=200, do_sample=True, top_k=5)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


## Source: my_qwen2-0.5b.ipynb


In [ ]:
import os

model_path = "./Qwen2-0.5B"

# 检查路径是否存在
if os.path.exists(model_path):
    print(f"路径 {model_path} 存在")
else:
    print(f"路径 {model_path} 不存在")

In [ ]:
import os
# 一定要在任何 torch/transformers 之前
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

In [ ]:
import os
import pandas as pd
import torch
import matplotlib.pyplot as plt
from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForSeq2Seq,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    TrainerCallback,
)
from peft import LoraConfig, TaskType, get_peft_model, PeftModel

In [ ]:
# Debug：确认 GPU 可见
print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("Using GPU:", torch.cuda.current_device(), torch.cuda.get_device_name(0))

In [ ]:
import torch
import torchvision
import torchaudio

print(torch.__version__)        # 查看 torch 版本
print(torchvision.__version__)  # 查看 torchvision 版本
print(torchaudio.__version__)   # 查看 torchaudio 版本


In [ ]:
# 加载并预处理诗歌数据集
def load_dataset():
    df = pd.read_json('poems_clean.json')
    df = df[['translation', 'content']].rename(columns={'translation': 'input', 'content': 'output'})
    ds = Dataset.from_pandas(df)
    return ds.train_test_split(test_size=0.1, seed=42)

ds = load_dataset()
train_ds = ds['train']
eval_ds = ds['test']
print(f"Train size: {len(train_ds)}, Eval size: {len(eval_ds)}")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    "./Qwen2-0.5B",
    use_fast=False,
    trust_remote_code=True,
    pad_token='<|endoftext|>'  # 明确设置填充token
)
tokenizer

In [ ]:
def process_func(example):
    MAX_LENGTH = 600
    # 构造三段式prompt
    prompt = (
        "<|im_start|>system\n你是一名古诗词诗人，擅长将白话文转换为古诗词。\n<|im_end|>\n"
        "<|im_start|>user\n{input}\n<|im_end|>\n<|im_start|>assistant\n"
    ).format(input=example['input'])
    
    # 编码prompt和响应
    p_tok = tokenizer(prompt, add_special_tokens=False)
    r_tok = tokenizer(example['output'], add_special_tokens=False)
    
    # 拼接输入
    input_ids = p_tok.input_ids + r_tok.input_ids + [tokenizer.pad_token_id]
    attention_mask = p_tok.attention_mask + r_tok.attention_mask + [1]
    labels = [-100] * len(p_tok.input_ids) + r_tok.input_ids + [tokenizer.pad_token_id]
    
    # 截断处理
    if len(input_ids) > MAX_LENGTH:
        input_ids = input_ids[:MAX_LENGTH]
        attention_mask = attention_mask[:MAX_LENGTH]
        labels = labels[:MAX_LENGTH]
    
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

# 应用处理
tokenized_ds = ds.map(process_func, remove_columns=ds['train'].column_names, batched=False)
tokenized_ds

In [ ]:
tokenizer.decode(tokenized_ds["train"][0]["input_ids"])


In [ ]:
tokenizer.decode(list(filter(lambda x: x != -100, tokenized_ds["train"][1]["labels"])))

In [ ]:
# 加载基础模型（确保本地路径正确）
model = AutoModelForCausalLM.from_pretrained(
    "./Qwen2-0.5B",
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True
)

model

In [ ]:
model.enable_input_require_grads() # 开启梯度检查点时，要执行该方法

In [ ]:
model.dtype

In [ ]:
# 添加LoRA适配器
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],  
    inference_mode=False, # 训练模式
    r=8, # Lora 秩
    lora_alpha=32,
    lora_dropout=0.1,  # Dropout 比例
    bias="none",
    base_model_name_or_path="./Qwen2-0.5B" 
)

In [ ]:
model = get_peft_model(model, lora_config)
lora_config

In [ ]:
model.print_trainable_parameters()

In [ ]:
# 一定要在任何 torch/transformers 之前
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

In [ ]:
# 接下来的训练设置
# ========= 修改后的 TrainingArguments =========
training_args = TrainingArguments(
    output_dir="D:\JZ\my-qwen2-0.5b-poetry-lora",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=4,
    logging_steps=10,
    eval_steps=50,
    num_train_epochs=50,              # 总共 50个 epoch
    save_strategy="epoch",           # 每个 epoch 结束时保存一次
    save_total_limit=3,              # 最多保留最近 3 个 checkpoint
    learning_rate=2e-5,
    gradient_checkpointing=True,
    fp16=True,
    report_to="none",
    label_names=["labels"],
    logging_dir="./logs",
    eval_strategy="epoch",     # 每个 epoch 评估一次
    load_best_model_at_end=True,    # 训练结束时加载最好的模型
    metric_for_best_model="eval_loss",  # 用 eval_loss 作为最优模型指标
)


In [ ]:
# 数据整理器
collator = DataCollatorForSeq2Seq(tokenizer, pad_to_multiple_of=8)  # 对齐显存

In [ ]:
class LossRecorderCallback(TrainerCallback):
    def __init__(self):
        self.train_losses = []
        self.eval_losses = []

    def on_epoch_end(self, args, state, control, **kwargs):
        train_loss = state.log_history[-1].get("loss", None)
        eval_loss = state.log_history[-1].get("eval_loss", None)

        if train_loss is not None:
            self.train_losses.append(train_loss)
        if eval_loss is not None:
            self.eval_losses.append(eval_loss)

    def plot_losses(self):
        plt.figure(figsize=(10, 5))
        plt.plot(self.train_losses, label='Train Loss')
        plt.plot(self.eval_losses, label='Eval Loss')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.title('Loss vs. Epoch')
        plt.legend()
        plt.grid(True)
        plt.show()

In [ ]:
# 实例化回调，指定保存路径（可改）
loss_callback = LossRecorderCallback()  # 注意这里有括号！表示实例化

In [ ]:
# 设置 Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["test"],
    data_collator=collator,
    callbacks=[
        #EarlyStoppingCallback(early_stopping_patience=3),  # 连续3个epoch没提升就停
        loss_callback,
    ]
)

In [ ]:
# 训练前显存清理
torch.cuda.empty_cache()

In [ ]:
# 启动训练
trainer.train()


In [ ]:
# 保存LoRA适配器（添加适配器保存逻辑）
model.save_pretrained("D:\JZ\my-qwen2-0.5b-poetry-lora")

In [ ]:
# 训练完成后画图
loss_callback.plot_losses()

In [ ]:
#保证目录下这个文件夹存在c
import os

print(os.path.abspath("D:\JZ\my-qwen2-0.5b-poetry-lora"))
print(os.listdir("D:\JZ\my-qwen2-0.5b-poetry-lora"))


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch
import re

# 加载 tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    "./Qwen2-0.5B",
    trust_remote_code=True
)

def generate_poetry(text):
    # 加载基础模型
    base_model = AutoModelForCausalLM.from_pretrained(
        "./Qwen2-0.5B",
        device_map="auto",
        torch_dtype=torch.bfloat16,
        trust_remote_code=True
    )

    # 合并 LoRA 权重
    model = PeftModel.from_pretrained(
        base_model,
        "D:\JZ\my-qwen2-0.5b-poetry-lora"
    )
    model = model.merge_and_unload()

    # 构造 prompt（Qwen 格式）
    prompt = (
    "<|im_start|>system\n"
    "你是一位精通古典诗词写作的文人，擅长用简洁典雅的语言表达深远意境。请将下面的白话精确地改写为一首非常古风的、风格高古、用词讲究、每句独立成行的古文，每句一行：<|im_end|>\n"
    f"<|im_start|>user\n请将下面的白话文精确地改写为一首风格高古、用词讲究的古文：\n\n{text}\n<|im_end|>\n"
    "<|im_start|>assistant\n"
)

    # 编码
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=300,       # 加长输出
        temperature=1.0,            # 提高创意度
        top_p=0.95,                 # 增加多样性
        repetition_penalty=1.05,   # 减少重复
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id
    )


    # 解码
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=False)

    print("\n===== 🔍 DEBUG 原始生成内容 =====\n")
    print(full_output)
    print("\n===== ✅ 提取后的古诗部分 =====\n")

    # 提取 assistant 部分（不依赖 <|im_end|>）
    match = re.search(r"assistant\n(.+?)(<\|im_end\|>|<\|endoftext\|>|$)", full_output, re.DOTALL)
    if match:
        poetry = match.group(1).strip()
        return "\n".join(poetry.splitlines()[:4])  # 返回前四行
    else:
        return "未能成功生成古诗"

# 示例调用
print(generate_poetry("我们都不喜欢秋天来到，因为树叶儿黄落，百草也都会慢慢凋零，这是寂寥的景色。"))


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch
import re

# 加载 tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    "./Qwen2-0.5B",
    trust_remote_code=True
)

def generate_poetry(text):
    # 加载基础模型
    base_model = AutoModelForCausalLM.from_pretrained(
        "./Qwen2-0.5B",
        device_map="auto",
        torch_dtype=torch.bfloat16,
        trust_remote_code=True
    )

    # 合并 LoRA 权重
    model = PeftModel.from_pretrained(
        base_model,
        "D:\JZ\my-qwen2-0.5b-poetry-lora"
    )
    model = model.merge_and_unload()

    # 构造 prompt（Qwen 格式）
    prompt = (
    "<|im_start|>system\n"
    "你是一位精通古典诗词写作的文人，擅长用简洁典雅的语言表达深远意境。请将下面的白话精确地改写为一首非常古风的、风格高古、用词讲究、每句独立成行的古文，每句一行：<|im_end|>\n"
    f"<|im_start|>user\n请将下面的白话文精确地改写为一首风格高古、用词讲究的古文：\n\n{text}\n<|im_end|>\n"
    "<|im_start|>assistant\n"
)

    # 编码
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=300,       # 加长输出
        temperature=1.0,            # 提高创意度
        top_p=0.95,                 # 增加多样性
        repetition_penalty=1.05,   # 减少重复
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id
    )


    # 解码
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=False)

    print("\n===== 🔍 DEBUG 原始生成内容 =====\n")
    print(full_output)
    print("\n===== ✅ 提取后的古诗部分 =====\n")

    # 提取 assistant 部分（不依赖 <|im_end|>）
    match = re.search(r"assistant\n(.+?)(<\|im_end\|>|<\|endoftext\|>|$)", full_output, re.DOTALL)
    if match:
        poetry = match.group(1).strip()
        return "\n".join(poetry.splitlines()[:4])  # 返回前四行
    else:
        return "未能成功生成古诗"

# 示例调用
print(generate_poetry("雪白玉润的莲藕，中间有精巧空明的窍孔，它的绿叶和红花铺散在水面上，互相映衬莲藕的生命延续不断，显示出无限意趣和欣欣向荣的生机，它之所以能够如此，全部奥秘都在莲籽的“苦心”当中"))

In [ ]:
from tqdm import tqdm
import torch
import pandas as pd
from transformers import TextStreamer
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import evaluate  # 新库

# —— 1. 加载 Tokenizer 与基础模型 —— #
BASE_MODEL_PATH = "./Qwen2-0.5B"
LORA_OUTPUT_PATH = "./model_adapter/qwen2-0.5b-poetry-lora"

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL_PATH,
    use_fast=False,
    trust_remote_code=True,
    pad_token='<|endoftext|>'
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_PATH,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True
)

# —— 2. 合并并加载 LoRA 权重 —— #
model = PeftModel.from_pretrained(base_model, LORA_OUTPUT_PATH)
model = model.merge_and_unload().eval()
device = next(model.parameters()).device


bleu_metric = evaluate.load("bleu")
rouge_metric = evaluate.load("rouge")

# 使用 TextStreamer 可选，控制生成打印
streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

# —— 1. 生成预测 & 收集参考 —— #
preds, refs, perplexities, inputs_cleaned = [], [], [], []  # 添加 inputs_cleaned 用于存储有效的 input

model.eval()
for sample in tqdm(ds['test'], desc="Evaluating"):
    src = sample['input']
    tgt = sample['output']

    # 如果 src 为 None，则跳过当前样本
    if src is None:
        print("Warning: Found None in input, skipping sample.")
        continue

    # 构造 prompt
    prompt = (
        "<|im_start|>system\n你是一名古诗词诗人，擅长将白话文转换为古诗词。\n<|im_end|>\n"
        "<|im_start|>user\n" + src + "\n<|im_end|>\n<|im_start|>assistant\n"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            streamer=None,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
        pred = tokenizer.decode(outputs[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True)

        # 计算困惑度
        encoded = tokenizer(pred, return_tensors="pt").to(model.device)
        with torch.no_grad():
            loss = model(**encoded, labels=encoded["input_ids"]).loss
        perplexity = torch.exp(loss).item()

    preds.append(pred)
    refs.append(tgt)
    perplexities.append(perplexity)
    inputs_cleaned.append(src)  # 保存有效的 input

# —— 2. 保存为CSV —— #
df = pd.DataFrame({
    "Input": inputs_cleaned,  # 使用处理过后的有效 inputs
    "Reference": refs,
    "Prediction": preds,
    "Perplexity": perplexities
})
df.to_csv("eval_results.csv", index=False)

# —— 3. 自动指标计算 —— #
bleu_result = bleu_metric.compute(predictions=[p.split() for p in preds],
                                  references=[[r.split()] for r in refs])
rouge_result = rouge_metric.compute(predictions=preds, references=refs)

print("BLEU:", bleu_result["bleu"])
print("ROUGE:", rouge_result)

# —— 4. 可视化 —— #
sns.set(style="whitegrid")
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

# 4.1 困惑度直方图
plt.figure(figsize=(8, 6))
sns.histplot(perplexities, bins=20, kde=True, color='skyblue')
plt.title("困惑度分布直方图", fontsize=14)
plt.xlabel("Perplexity")
plt.ylabel("样本数量")
plt.tight_layout()
plt.savefig("perplexity_histogram.png")
plt.show()

# 4.2 困惑度趋势折线图
plt.figure(figsize=(10, 6))
plt.plot(perplexities, marker='o', linestyle='-', color='orange')
plt.title("样本困惑度变化趋势", fontsize=14)
plt.xlabel("样本序号")
plt.ylabel("Perplexity")
plt.tight_layout()
plt.savefig("perplexity_trend.png")
plt.show()

# 4.3 指标柱状图
scores = {
    "BLEU": bleu_result["bleu"],
    "ROUGE-1": rouge_result["rouge1"],
    "ROUGE-2": rouge_result["rouge2"],
    "ROUGE-L": rouge_result["rougeL"]
}
plt.figure(figsize=(8, 6))
sns.barplot(x=list(scores.keys()), y=list(scores.values()), palette="Blues_d")
plt.title("自动评价指标得分", fontsize=14)
plt.ylabel("Score")
plt.ylim(0, 1)
plt.tight_layout()
plt.savefig("metrics_barplot.png")
plt.show()



In [ ]:
from tqdm import tqdm
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer
from peft import PeftModel
import seaborn as sns
import matplotlib.pyplot as plt
import evaluate
import numpy as np

# 路径
BASE_MODEL_PATH = "./Qwen2-0.5B"
LORA_OUTPUT_PATH = "./model_adapter/qwen2-0.5b-poetry-lora"

# 加载 tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL_PATH,
    use_fast=False,
    trust_remote_code=True,
    pad_token='<|endoftext|>'
)

# 加载并合并模型
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_PATH,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True
)
model = PeftModel.from_pretrained(base_model, LORA_OUTPUT_PATH)
model = model.merge_and_unload().eval()
device = next(model.parameters()).device

# 加载评估指标
bleu_metric = evaluate.load("bleu")
rouge_metric = evaluate.load("rouge")

# 用来存结果
preds, refs, perplexities, inputs_cleaned = [], [], [], []

# 模型评估
for sample in tqdm(ds['test'], desc="Evaluating"):
    src = sample['input']
    tgt = sample['output']

    if not src or not tgt:
        continue

    # Prompt 构造
    prompt = (
        "<|im_start|>system\n你是一名古诗词诗人，擅长将白话文转换为古诗词。\n<|im_end|>\n"
        f"<|im_start|>user\n{src}\n<|im_end|>\n<|im_start|>assistant\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    # 解码并保存预测
    pred = tokenizer.decode(outputs[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True)
    preds.append(pred)
    refs.append(tgt)
    inputs_cleaned.append(src)

    # ➤ 用 reference 来计算 perplexity
    tgt_encoded = tokenizer(tgt, return_tensors="pt").to(device)
    with torch.no_grad():
        loss = model(**tgt_encoded, labels=tgt_encoded["input_ids"]).loss
        perplexity = torch.exp(loss).item()
    perplexities.append(perplexity)

# —— 保存结果 —— #
df = pd.DataFrame({
    "Input": inputs_cleaned,
    "Reference": refs,
    "Prediction": preds,
    "Perplexity": perplexities
})
df.to_csv("eval_results.csv", index=False)

# —— 自动指标 —— #
bleu_result = bleu_metric.compute(predictions=[p.split() for p in preds],
                                  references=[[r.split()] for r in refs])
rouge_result = rouge_metric.compute(predictions=preds, references=refs)

print("BLEU:", bleu_result["bleu"])
print("ROUGE:", rouge_result)

# —— 可视化 —— #
sns.set(style="whitegrid")
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

# 困惑度直方图
plt.figure(figsize=(8, 6))
sns.histplot(perplexities, bins=20, kde=True, color='skyblue')
plt.title("困惑度分布直方图", fontsize=14)
plt.xlabel("Perplexity")
plt.ylabel("样本数量")
plt.tight_layout()
plt.savefig("perplexity_histogram.png")
plt.show()

# 困惑度趋势图
plt.figure(figsize=(10, 6))
plt.plot(perplexities, marker='o', linestyle='-', color='orange')
plt.title("样本困惑度变化趋势", fontsize=14)
plt.xlabel("样本序号")
plt.ylabel("Perplexity")
plt.tight_layout()
plt.savefig("perplexity_trend.png")
plt.show()

# BLEU / ROUGE 图表
scores = {
    "BLEU": bleu_result["bleu"],
    "ROUGE-1": rouge_result["rouge1"],
    "ROUGE-2": rouge_result["rouge2"],
    "ROUGE-L": rouge_result["rougeL"]
}
plt.figure(figsize=(8, 6))
sns.barplot(x=list(scores.keys()), y=list(scores.values()), palette="Blues_d")
plt.title("自动评价指标得分", fontsize=14)
plt.ylabel("Score")
plt.ylim(0, 1)
plt.tight_layout()
plt.savefig("metrics_barplot.png")
plt.show()
